# DermaScan SCIN Training
This notebook trains a skin condition detection model using the **SCIN (Skin Condition Image Network)** dataset.

**Dataset Source**: `gs://dx-scin-public-data/` (Google Research)
**Model Architecture**: EfficientNetB0 (Transfer Learning)
**Resolution**: 224x224 (1:1 aspect ratio)


## 1. Setup & Data Downloading
Downloading metadata and images from the public GCS bucket.

In [ ]:
!pip install -q -U transformers

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import ast

# 1. Download Metadata
!mkdir -p scin_data
!gsutil -m cp gs://dx-scin-public-data/dataset/scin_cases.csv scin_data/
!gsutil -m cp gs://dx-scin-public-data/dataset/scin_labels.csv scin_data/

# 2. Load Metadata
cases_df = pd.read_csv('scin_data/scin_cases.csv')
labels_df = pd.read_csv('scin_data/scin_labels.csv')

# Merge on case_id
df = pd.merge(cases_df, labels_df, on='case_id')

print(f"Total cases: {len(df)}")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://dx-scin-public-data/dataset/scin_cases.csv...
/ [1/1 files][  1.2 MiB/  1.2 MiB] 100% Done                                    
Operation completed over 1 objects/1.2 MiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://dx-scin-public-data/dataset/scin_labels.csv...
/ [1/1 files][874.7 KiB/874.7 KiB] 100% Done                                    
Operation completed over 1 objects/874.7 KiB.                                    
Total cases: 5033


## 2. Label Processing
We extract the primary condition from the `weighted_skin_condition_label` dictionary and handle healthy skin cases.

In [ ]:
def get_primary_label(row):
    # 1. Check if user reported LOOKS_HEALTHY
    if row['related_category'] == 'LOOKS_HEALTHY':
        return 'Healthy Skin'

    # 2. Check if dermatologist marked as ungradable due to no pathology
    # (In SCIN, this is often indicated by a lack of weighted labels despite being a valid case)
    label_dict_str = row['weighted_skin_condition_label']
    if pd.isna(label_dict_str) or label_dict_str == '{}':
        return 'Healthy Skin'

    try:
        # Parse the dictionary-like string
        label_dict = ast.literal_eval(label_dict_str)
        if not label_dict:
            return 'Healthy Skin'
        # Get condition with highest weight
        return max(label_dict, key=label_dict.get)
    except:
        return 'Unknown'

df['primary_label'] = df.apply(get_primary_label, axis=1)

# Filter for top 10 conditions + Healthy Skin
TOP_10_CONDITIONS = [
    'Eczema', 'Acne', 'Psoriasis', 'Tinea', 'Herpes Simplex',
    'Impetigo', 'Rosacea', 'Insect Bite', 'Urticaria', 'Folliculitis'
]
ALLOWED_LABELS = TOP_10_CONDITIONS + ['Healthy Skin']

train_df = df[df['primary_label'].isin(ALLOWED_LABELS)].copy()
print(f"Cases after filtering: {len(train_df)}")
print(train_df['primary_label'].value_counts())

Cases after filtering: 3507
primary_label
Healthy Skin      2134
Eczema             476
Urticaria          202
Insect Bite        168
Folliculitis       135
Psoriasis          101
Tinea               88
Impetigo            68
Acne                58
Herpes Simplex      50
Rosacea             27
Name: count, dtype: int64


## 3. Image Downloading
We only download images for the cases we filtered. We'll use a local directory for storage.

In [ ]:
# SCIN uses image_1_path, image_2_path, etc.
image_paths = []
labels = []
ids = []

for idx, row in train_df.iterrows():
    for i in range(1, 4):
        path = row.get(f'image_{i}_path')
        if pd.notna(path):
            image_paths.append(path)
            labels.append(row['primary_label'])
            ids.append(row['case_id'])

image_df = pd.DataFrame({'case_id': ids, 'gcs_path': image_paths, 'label': labels})

# For this notebook, we'll download a subset or use a generator that pulls from GCS if possible.
# To keep it simple, we'll download the images needed.
print(f"Total images to download: {len(image_df)}")

# Caution: Downloading 10k images might take time and space.
# In a real Colab, you would use:
# !gsutil -m cp -r gs://dx-scin-public-data/dataset/images scin_data/images

Total images to download: 7175


## 4. Model Building (EfficientNetB0)
We use transfer learning with EfficientNetB0.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2

IMG_SIZE = 224
NUM_CLASSES = len(ALLOWED_LABELS)

def build_model():
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # MobileNetV2 expects pixel values in the range [-1, 1]
    x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(inputs)

    # Load pre-trained MobileNetV2 (Nimble, fast convergence)
    base_model = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_tensor=x
    )
    base_model.trainable = False # Start frozen
    base_model._name = 'mobilenetv2_backbone'

    # Classification head with a "bridge" layer
    x = GlobalAveragePooling2D()(base_model.output)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    # Intermediate representation layer
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    predictions = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)

    # Compile with a smaller initial learning rate to handle class weights safely
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
    )
    return model

model = build_model()
model.summary()


/tmp/ipykernel_77691/1536188180.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis

 Total params: 2,594,891 (9.90 MB)

 Trainable params: 333,835 (1.27 MB)

 Non-trainable params: 2,261,056 (8.63 MB)

## 5. Training
Train for a few epochs frozen, then unfreeze for fine-tuning.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

print("Starting MobileNetV2 Training (Optimized for Recall)...")

# Prepare data arrays
X = image_df['gcs_path'].values
y = image_df['label'].values

# Encode labels into one-hot format
lb = LabelBinarizer()
y_encoded = lb.fit_transform(y)

# Compute class weights to handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weight_dict)

# Define data augmentation
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal_and_vertical'),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
  tf.keras.layers.RandomBrightness(0.2),
  tf.keras.layers.RandomContrast(0.2)
])

def load_and_preprocess_image(path, label):
    gcs_path = tf.strings.join(['gs://dx-scin-public-data/', path])
    img = tf.io.read_file(gcs_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def prepare_dataset(X_split, y_split, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((X_split, y_split))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

    # Ignore missing files (404 errors) from the GCS bucket
    ds = ds.ignore_errors()

    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

# Create a single train/validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.15, stratify=y, random_state=42)

# Create datasets
train_ds = prepare_dataset(X_train, y_train, augment=True)
val_ds = prepare_dataset(X_val, y_val, augment=False)

# Build a fresh model
model = build_model()

print("1. Training top layers (frozen base)...")
# Train the new dense layers briefly
model.fit(train_ds, validation_data=val_ds, epochs=4, class_weight=class_weight_dict)

Starting MobileNetV2 Training (Optimized for Recall)...
Class Weights: {0: np.float64(5.6230407523510975), 1: np.float64(0.6188545799551493), 2: np.float64(2.272727272727273), 3: np.float64(0.15427453341360625), 4: np.float64(6.5227272727272725), 5: np.float64(4.831649831649831), 6: np.float64(1.8068496600352557), 7: np.float64(2.951460304401481), 8: np.float64(12.307032590051458), 9: np.float64(3.2613636363636362), 10: np.float64(1.553030303030303)}


/tmp/ipykernel_77691/1536188180.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


1. Training top layers (frozen base)...
Epoch 1/4
    191/Unknown 132s 575ms/step - accuracy: 0.0903 - loss: 3.6235 - recall: 0.0370

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


191/191 ━━━━━━━━━━━━━━━━━━━━ 166s 754ms/step - accuracy: 0.0914 - loss: 3.3874 - recall: 0.0362 - val_accuracy: 0.0826 - val_loss: 2.7193 - val_recall: 0.0019
Epoch 2/4
138/191 ━━━━━━━━━━━━━━━━━━━━ 28s 533ms/step - accuracy: 0.0927 - loss: 2.9892 - recall: 0.0378

KeyboardInterrupt: 

In [ ]:
print("2. Fine-tuning base model...")
# Unfreeze the top layers of the base model for fine-tuning
model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])

# Callbacks: Monitor 'val_recall' instead of accuracy or loss
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_recall', mode='max', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_recall', mode='max', factor=0.5, patience=2)
]

history = model.fit(train_ds, validation_data=val_ds, epochs=15,
                    class_weight=class_weight_dict, callbacks=callbacks)

# Extract best validation recall from history
val_recall = max(history.history['val_recall'])
print(f"\nBest Validation Recall: {val_recall:.4f}")

In [ ]:
# --- Evaluate Model (Confusion Matrix & Classification Report) ---
print("\nEvaluating on Validation Data...")
y_true = []
y_pred_probs = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_pred = np.argmax(y_pred_probs, axis=1)
class_names = lb.classes_

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Validation Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Save model
model_path = 'dermascan.keras'
model.save(model_path)
print(f"✅ Model saved to {model_path}")


In [ ]:
# Mount Google Drive and save model there
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/dermascan.keras', '/content/drive/MyDrive/dermascan.keras')
print('✅ Model saved to Google Drive → My Drive/dermascan.keras')

## Version 2: Advanced Pipeline (HAM10000, Cropping, Resampling, MixUp)

This section contains the advanced techniques to improve our baseline 14% accuracy.

### 1. Download HAM10000 Dataset
**Note:** You must upload your `kaggle.json` file to the Colab environment first to use the Kaggle API.

In [8]:
!pip install -q kaggle
import os
import getpass

# Set Kaggle environment variables directly
os.environ['KAGGLE_USERNAME'] = input('Enter your Kaggle Username: ')
os.environ['KAGGLE_KEY'] = getpass.getpass('Enter your Kaggle Key: ')

# Download and extract the dataset
print("Downloading HAM10000...")
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
print("Extracting...")
!unzip -q -o skin-cancer-mnist-ham10000.zip -d ham10000_data/
print("Done!")

Enter your Kaggle Username: ethanharter
Enter your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:26<00:00, 210MB/s]

Extracting...
Done!


### 2. Targeted Lesion Cropping (OpenCV)
Uses thresholding to find the darkest region (often the lesion) and crops around it.

In [9]:
import cv2
import numpy as np
from PIL import Image

def auto_crop_lesion(img_array, padding=20):
    # Convert to grayscale
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

    # Blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (15, 15), 0)

    # Threshold to isolate the darker lesion from lighter skin
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return img_array # Return original if no contour found

    # Find the largest contour (assuming it's the lesion)
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    # Add padding
    x = max(0, x - padding)
    y = max(0, y - padding)
    w = min(img_array.shape[1] - x, w + 2*padding)
    h = min(img_array.shape[0] - y, h + 2*padding)

    return img_array[y:y+h, x:x+w]

### 3. Oversampling Minority Classes
Instead of just using class weights, we physically duplicate rows of the minority classes so the dataset is perfectly balanced.

In [10]:
import pandas as pd

# Assuming `image_df` is our base dataframe from earlier
max_size = image_df['label'].value_counts().max()

# Resample each class to have the same number of samples as the majority class
balanced_dfs = []
for class_name, group in image_df.groupby('label'):
    balanced_group = group.sample(max_size, replace=True, random_state=42)
    balanced_dfs.append(balanced_group)

balanced_image_df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
print("New balanced dataset size:", len(balanced_image_df))
print(balanced_image_df['label'].value_counts())

New balanced dataset size: 46508
label
Impetigo          4228
Herpes Simplex    4228
Rosacea           4228
Tinea             4228
Insect Bite       4228
Psoriasis         4228
Acne              4228
Eczema            4228
Healthy Skin      4228
Urticaria         4228
Folliculitis      4228
Name: count, dtype: int64


### 4. Advanced Augmentation (MixUp & CutMix)
We use `keras_cv` to apply state-of-the-art data augmentation.

In [11]:
!pip install -q keras-cv
import keras_cv
import tensorflow as tf

# Define CutMix and MixUp layers
cut_mix = keras_cv.layers.CutMix()
mix_up = keras_cv.layers.MixUp()

def advanced_augment(images, labels):
    # Apply KerasCV augmentations (operates on batches)
    # We randomly apply either CutMix or MixUp
    inputs = {"images": images, "labels": labels}

    # 50% chance of CutMix, 50% chance of MixUp
    if tf.random.uniform([]) > 0.5:
        outputs = cut_mix(inputs)
    else:
        outputs = mix_up(inputs)

    return outputs['images'], outputs['labels']

# Example of how to map this to your dataset (requires batching first!):
# train_ds = train_ds.batch(32).map(advanced_augment, num_parallel_calls=tf.data.AUTOTUNE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 35.7 MB/s eta 0:00:00


### 5. Train & Save Version 2 Model
Now we put all the advanced pieces together: we use the balanced dataset, apply the advanced KerasCV augmentations to the batches, train the model, and save it.

In [13]:
import tensorflow as tf

print("Assembling Version 2 Pipeline...")

# 1. Prepare data from our balanced dataframe
X_v2 = balanced_image_df['gcs_path'].values
y_v2 = balanced_image_df['label'].values

# Encode labels and cast to float32 for MixUp/CutMix
y_encoded_v2 = lb.transform(y_v2).astype('float32')

# Train/Val Split
X_train_v2, X_val_v2, y_train_v2, y_val_v2 = train_test_split(
    X_v2, y_encoded_v2, test_size=0.15, stratify=y_v2, random_state=42
)

# 2. Build Datasets
# Using the load_and_preprocess_image function we defined in Version 1
train_ds_v2 = tf.data.Dataset.from_tensor_slices((X_train_v2, y_train_v2))
train_ds_v2 = train_ds_v2.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_v2 = train_ds_v2.ignore_errors()

# CRITICAL: We must batch the dataset BEFORE applying CutMix/MixUp
train_ds_v2 = train_ds_v2.batch(32)
train_ds_v2 = train_ds_v2.map(advanced_augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_v2 = train_ds_v2.prefetch(tf.data.AUTOTUNE)

# Validation Dataset (No augmentation)
val_ds_v2 = tf.data.Dataset.from_tensor_slices((X_val_v2, y_val_v2))
val_ds_v2 = val_ds_v2.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_v2 = val_ds_v2.ignore_errors().batch(32).prefetch(tf.data.AUTOTUNE)

# 3. Build & Compile Model
model_v2 = build_model()
# Unfreeze base model for fine-tuning
model_v2.trainable = True

model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

# 4. Train Model (No class_weights needed since we balanced the data via resampling!)
print("Training Version 2 Model with Advanced Augmentation & Resampling...")
history_v2 = model_v2.fit(train_ds_v2, validation_data=val_ds_v2, epochs=10)

# 5. Save the upgraded model
model_v2_path = 'dermascan_v2.keras'
model_v2.save(model_v2_path)
print(f"\n✅ Version 2 Model successfully saved to {model_v2_path}")

Assembling Version 2 Pipeline...


/tmp/ipykernel_77691/1536188180.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Training Version 2 Model with Advanced Augmentation & Resampling...
Epoch 1/10
   1235/Unknown 595s 472ms/step - accuracy: 0.1499 - loss: 3.2647 - recall: 0.0576

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1235/1235 ━━━━━━━━━━━━━━━━━━━━ 712s 566ms/step - accuracy: 0.1832 - loss: 2.9983 - recall: 0.0685 - val_accuracy: 0.4532 - val_loss: 1.6391 - val_recall: 0.1856
Epoch 2/10
1235/1235 ━━━━━━━━━━━━━━━━━━━━ 645s 522ms/step - accuracy: 0.2549 - loss: 2.4709 - recall: 0.0824 - val_accuracy: 0.5919 - val_loss: 1.3977 - val_recall: 0.2530
Epoch 3/10
1235/1235 ━━━━━━━━━━━━━━━━━━━━ 642s 520ms/step - accuracy: 0.2941 - loss: 2.2471 - recall: 0.0811 - val_accuracy: 0.6355 - val_loss: 1.2933 - val_recall: 0.2788
Epoch 4/10
1235/1235 ━━━━━━━━━━━━━━━━━━━━ 622s 504ms/step - accuracy: 0.3225 - loss: 2.1099 - recall: 0.0800 - val_accuracy: 0.6623 - val_loss: 1.2310 - val_recall: 0.2987
Epoch 5/10
1235/1235 ━━━━━━━━━━━━━━━━━━━━ 645s 522ms/step - accuracy: 0.3470 - loss: 2.0178 - recall: 0.0813 - val_accuracy: 0.6881 - val_loss: 1.1785 - val_recall: 0.3326
Epoch 6/10
1235/1235 ━━━━━━━━━━━━━━━━━━━━ 640s 518ms/step - accuracy: 0.3594 - loss: 1.9764 - recall: 0.0820 - val_accuracy: 0.7056 - val_loss: 1.1403 

In [14]:
# Mount Google Drive and save model there
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/dermascan.keras', '/content/drive/MyDrive/dermascan.keras')
print('✅ Model saved to Google Drive → My Drive/dermascan.keras')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model saved to Google Drive → My Drive/dermascan.keras
